# Stage 2 - Feature Engineering

Build a **leakage-safe** feature table for played matches (training) and the
unplayed 2026 World Cup fixtures, from the Stage 1 interim files.

Features: temporal as-of Elo (`home_elo`, `away_elo`, `elo_diff`), rolling
last-5 form (winrate / goals for / goals against / goal diff per side), and
context (`neutral`, `tournament_weight`).

**No models are trained and no tournament is simulated here.**

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, features

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

## 1. Load cleaned interim inputs

In [2]:
played = pd.read_parquet(config.INTERIM_FILES["results_played"])
fixtures = pd.read_parquet(config.INTERIM_FILES["fixtures_2026"])
elo = pd.read_parquet(config.INTERIM_FILES["eloratings_clean"])
print("played  :", played.shape)
print("fixtures:", fixtures.shape)
print("elo     :", elo.shape)

played  : (49400, 9)
fixtures: (72, 7)
elo     : (6646, 4)


## 2. Targets: home_win / draw / away_win

In [3]:
played_t = features.add_targets(played)
print(played_t["result"].value_counts())
# Each match has exactly one outcome flag set.
assert (played_t[["home_win", "draw", "away_win"]].sum(axis=1) == 1).all()
played_t[["home_team", "away_team", "home_score", "away_score", "result", "home_win", "draw", "away_win"]].head()

result
home_win    24209
away_win    13958
draw        11233
Name: count, dtype: int64


,home_team,away_team,home_score,away_score,result,home_win,draw,away_win
0,Scotland,England,0,0,draw,0,1,0
1,England,Scotland,4,2,home_win,1,0,0
2,Scotland,England,2,1,home_win,1,0,0
3,England,Scotland,2,2,draw,0,1,0
4,Scotland,England,3,0,home_win,1,0,0


## 3. Elo temporal as-of join (strictly before match date)

We attach each team's latest Elo rating dated *strictly before* the match
(`allow_exact_matches=False`). The Elo `change` column is never used.

In [4]:
played_e, fixtures_e = features.add_elo_features(played_t, fixtures, elo)
played_e[["date", "home_team", "away_team", "home_elo", "away_elo", "elo_diff"]].tail()

,date,home_team,away_team,home_elo,away_elo,elo_diff
49395,2026-06-09,Armenia,Moldova,1389.0,NaN,NaN
49396,2026-06-09,Argentina,Iceland,2113.0,1566.0,547.0
49397,2026-06-09,Angola,Central African Republic,1566.0,NaN,NaN
49398,2026-06-09,Ethiopia,Malawi,1264.0,1252.0,12.0
49399,2026-06-09,Iraq,Venezuela,1582.0,1715.0,-133.0


### Leakage check: attached Elo must predate the match

For a sample team, confirm the joined `home_elo` equals the most recent Elo
rating strictly before the match date (and never on/after it).

In [5]:
sample = played_e[played_e["home_team"] == "Brazil"].dropna(subset=["home_elo"]).iloc[-1]
prior = elo[(elo["team"] == "Brazil") & (elo["date"] < sample["date"])].sort_values("date")
print("match date     :", sample["date"].date())
print("joined home_elo:", sample["home_elo"])
print("expected (last prior):", prior["rating"].iloc[-1])
assert sample["home_elo"] == prior["rating"].iloc[-1]
print("OK - Elo is strictly pre-match")

match date     : 2026-06-06
joined home_elo: 1979.0
expected (last prior): 1979.0
OK - Elo is strictly pre-match


## 4. Rolling form features (only matches strictly before the current match)

In [6]:
played_f, fixtures_f = features.add_form_features(played_e, fixtures_e, config.FORM_WINDOW)
form_cols = [
    "home_winrate_last5", "away_winrate_last5",
    "home_goals_for_last5", "away_goals_for_last5",
    "home_goals_against_last5", "away_goals_against_last5",
    "home_goal_diff_last5", "away_goal_diff_last5",
]
played_f[["date", "home_team", "away_team"] + form_cols].tail()

,date,home_team,away_team,home_winrate_last5,away_winrate_last5,home_goals_for_last5,away_goals_for_last5,home_goals_against_last5,away_goals_against_last5,home_goal_diff_last5,away_goal_diff_last5
49395,2026-06-09,Armenia,Moldova,0.0,0.0,0.6,1.0,2.8,2.6,-2.2,-1.6
49396,2026-06-09,Argentina,Iceland,1.0,0.0,3.4,0.6,0.2,2.0,3.2,-1.4
49397,2026-06-09,Angola,Central African Republic,0.2,0.2,1.2,0.8,1.2,2.4,0.0,-1.6
49398,2026-06-09,Ethiopia,Malawi,0.4,0.2,0.6,0.2,1.4,0.6,-0.8,-0.4
49399,2026-06-09,Iraq,Venezuela,0.4,0.4,0.8,1.2,1.0,1.0,-0.2,0.2


### Leakage check: form uses only prior matches

Manually recompute home-team last-5 winrate for one match from raw history and
compare to the engineered value. The current match must be excluded.

In [7]:
row = played_f[played_f["home_team"] == "Brazil"].dropna(subset=["home_winrate_last5"]).iloc[-1]
team, dt = row["home_team"], row["date"]
hist = played_t[
    ((played_t["home_team"] == team) | (played_t["away_team"] == team))
    & (played_t["date"] < dt)  # strictly before -> current match excluded
].sort_values("date").tail(config.FORM_WINDOW)
wins = (
    ((hist["home_team"] == team) & (hist["home_score"] > hist["away_score"]))
    | ((hist["away_team"] == team) & (hist["away_score"] > hist["home_score"]))
).mean()
print("engineered winrate:", round(row["home_winrate_last5"], 4))
print("manual    winrate :", round(wins, 4))
assert abs(row["home_winrate_last5"] - wins) < 1e-9
print("OK - form excludes the current match")

engineered winrate: 0.6
manual    winrate : 0.6
OK - form excludes the current match


## 5. Context features: neutral + tournament_weight

In [8]:
played_c = features.add_context_features(played_f)
weights = (
    played_c[["tournament", "tournament_weight"]]
    .drop_duplicates()
    .sort_values("tournament_weight", ascending=False)
)
weights.head(12)

,tournament,tournament_weight
1490,FIFA World Cup,1.00
43370,CONIFA World Cup qualification,0.85
1771,FIFA World Cup qualification,0.85
18556,Confederations Cup,0.85
480,Copa América,0.80
5084,UEFA Euro,0.80
4321,AFC Asian Cup,0.80
4399,African Cup of Nations,0.80
17839,Gold Cup,0.80
29,British Home Championship,0.75


## 6 & 7. Build, save, and summarize the feature tables

`features.run()` writes `data/processed/train_features.parquet` and
`data/processed/fixtures_2026_features.parquet`, then prints the summary.

In [9]:
train, fix = features.run()

STAGE 2 FEATURE SUMMARY
train_features rows         : 49,400
fixtures_2026_features rows : 72

--- target distribution (train) ---
result
home_win    24209
away_win    13958
draw        11233

--- missing values: train_features ---
date                            0
home_team                       0
away_team                       0
tournament                      0
home_elo                    12338
away_elo                    12306
elo_diff                    20218
home_winrate_last5            141
away_winrate_last5            195
home_goals_for_last5          141
away_goals_for_last5          195
home_goals_against_last5      141
away_goals_against_last5      195
home_goal_diff_last5          141
away_goal_diff_last5          195
neutral                         0
tournament_weight               0
result                          0
home_win                        0
draw                            0
away_win                        0

--- missing values: fixtures_2026_features ---
date  

In [10]:
for key, path in config.PROCESSED_FILES.items():
    print(f"{key:<24} {'exists' if path.exists() else 'MISSING':<8} {path.name}")
fix.head()

train_features           exists   train_features.parquet
fixtures_2026_features   exists   fixtures_2026_features.parquet


,date,home_team,away_team,tournament,home_elo,away_elo,elo_diff,home_winrate_last5,away_winrate_last5,home_goals_for_last5,away_goals_for_last5,home_goals_against_last5,away_goals_against_last5,home_goal_diff_last5,away_goal_diff_last5,neutral,tournament_weight
0,2026-06-11,Mexico,South Africa,FIFA World Cup,1835.0,NaN,NaN,0.6,0.2,1.8,0.8,0.4,1.0,1.4,-0.2,0,1.0
1,2026-06-11,South Korea,Czech Republic,FIFA World Cup,NaN,NaN,NaN,0.6,0.6,1.4,3.4,1.0,1.2,0.4,2.2,1,1.0
2,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup,1802.0,NaN,NaN,0.4,0.0,1.4,0.8,0.6,0.8,0.8,0.0,0,1.0
3,2026-06-12,United States,Paraguay,FIFA World Cup,NaN,1833.0,NaN,0.4,0.6,2.2,1.8,2.4,1.0,-0.2,0.8,0,1.0
4,2026-06-13,Qatar,Switzerland,FIFA World Cup,1427.0,1897.0,-470.0,0.0,0.2,0.2,1.8,1.2,1.4,-1.0,0.4,1,1.0
